# 00 · Data Acquisition

Download public-domain golden age comics from the [Digital Comic Museum](https://digitalcomicmuseum.com/index.php?cid=110), extract their pages, and build a manifest CSV.

**Comics used (all public domain):**
Add direct CBZ download URLs to `COMIC_URLS` below. You can find download links by browsing the category page and clicking through to each comic's download page.

In [ ]:
from pathlib import Path
import pandas as pd
import sys
sys.path.insert(0, str(Path().resolve()))

from utils.downloader import download_comic
from utils.extractor import extract_comic

RAW_DIR   = Path("data/raw")
PAGES_DIR = Path("data/pages")
MAX_PAGES = 30  # set lower to reduce cost during testing

## Define comics to download

Populate `COMIC_URLS` with `(filename, url)` tuples.  
Direct download links look like: `https://digitalcomicmuseum.com/download.php?...`

In [ ]:
# Add (filename, url) pairs — get URLs from the DCM download pages
COMIC_URLS = [
    # ("comic_filename.cbz", "https://..."),
]

if not COMIC_URLS:
    print("⚠️  No comic URLs defined. Add entries to COMIC_URLS above.")
    print("   Browse https://digitalcomicmuseum.com/index.php?cid=110 to find download links.")

In [ ]:
# Download archives
archive_paths = []
for filename, url in COMIC_URLS:
    path = download_comic(url, RAW_DIR, filename)
    archive_paths.append(path)
print(f"Downloaded {len(archive_paths)} archives.")

## Extract pages

In [ ]:
all_pages = []
for archive in archive_paths:
    pages = extract_comic(archive, PAGES_DIR)
    # Honour MAX_PAGES per comic to control cost
    pages = pages[:MAX_PAGES]
    all_pages.extend(pages)
    print(f"  {archive.name}: {len(pages)} pages")

print(f"\nTotal pages: {len(all_pages)}")

## Build and save manifest

In [ ]:
manifest = pd.DataFrame(all_pages)
manifest["page_id"] = manifest["comic_slug"] + "_p" + manifest["page_num"].astype(str).str.zfill(3)
manifest.to_csv("data/manifest.csv", index=False)
print(f"Saved manifest with {len(manifest)} rows.")
manifest.head()

In [ ]:
# Quick sanity check — display first page of each comic
from IPython.display import display, Image as IPyImage
from PIL import Image

for slug, grp in manifest.groupby("comic_slug"):
    first = grp.iloc[0]
    print(f"\n{first['comic_title']} — {len(grp)} pages")
    img = Image.open(first["file_path"])
    img.thumbnail((300, 400))
    display(img)